# 07 · Funktionen

**Lernziele**

- Du definierst eigene Funktionen mit Parametern und Rückgabewerten.
- Du weißt, wann eine Funktion `None` liefert, und nutzt `None` gezielt für „nichts gefunden“.
- Du nutzt Standardwerte, Schlüsselwort-Argumente und beliebig viele Argumente.
- Du dokumentierst Funktionen mit Docstrings und Type Hints.
- Du weißt, in welchem Namensraum Python einen Namen findet: lokal, global oder in den builtins.
- Du übergibst Funktionen als Werte und schreibst kleine Lambdas.
- Du löst Probleme rekursiv und erkennst, wann das sinnvoll ist.
- Du verstehst, wie `for` mit Iteratoren arbeitet, und schreibst Generatoren mit `yield`.

> Grundlage: Inden, *Einstieg in Python*, Kapitel 2, 6, 8 und 16.

## Warum Funktionen?

Funktionen geben einem Stück Code einen **Namen**. Das macht Programme kürzer, lesbarer und testbar. Faustregel: Kopierst du Code ein zweites Mal, gehört er in eine Funktion.

## Funktionen definieren

In [ ]:
def greet(name):
    print(f"Hallo {name}!")


greet("Ada")
greet("Linus")

Mit `return` liefert eine Funktion ein Ergebnis zurück. Ohne `return` liefert sie `None`.

In [ ]:
def gross_price(net_price, vat_rate):
    return net_price * (1 + vat_rate)


price = gross_price(100, 0.19)
print(f"{price:.2f} €")

![Der Aufruf gross_price(100, 0.19) springt in die Funktion, die Parameter net_price und vat_rate verweisen auf 100 und 0.19, return liefert 119.0 zurück, und price verweist danach auf 119.0.](../assets/07_funktionsaufruf.png)

> `print` zeigt etwas an, `return` gibt etwas zurück. Nur mit `return` kannst du mit dem Ergebnis weiterrechnen.

**Kurzübung 1 (Eigene Summenfunktion):** Schreibe `my_sum(numbers)`, die die Summe einer Zahlenliste mit einer `for`-Schleife berechnet und zurückgibt, ohne das eingebaute `sum`. Teste sie mit `prices` und vergleiche das Ergebnis mit `sum(prices)`. Welches Ergebnis liefert `my_sum([])`, und warum?

In [ ]:
prices = [4.5, 12.0, 3.2, 8.9]

In [ ]:
# Deine Lösung


## `None` als Rückgabewert

`None` steht für „kein Wert“; wie du darauf prüfst, steht in Notebook 03 (Abschnitt „None und Vergleiche“). Bei Funktionen begegnet dir `None` an drei Stellen.

Erstens liefert eine Funktion ohne `return` (oder mit leerem `return`) `None`. Typischer Fehler: Die Funktion druckt das Ergebnis, statt es zurückzugeben:

In [ ]:
def add_vat(net_price):
    print(f"{net_price * 1.19:.2f} €")  # druckt nur, gibt nichts zurück


total = add_vat(100)
print(total)  # None: mit total lässt sich nicht weiterrechnen

Zweitens liefern Methoden, die ein Objekt selbst verändern, `None`. Das kennst du aus Notebook 06, Kurzübung 2: `sort()` sortiert die Liste an Ort und Stelle; wer das Ergebnis wieder zuweist, verliert die Liste. `sorted()` liefert dagegen eine neue Liste:

In [ ]:
items = [3, 1, 2]
items = items.sort()
print(items)  # None: die Liste ist weg

items = [3, 1, 2]
print(sorted(items))  # neue, sortierte Liste
print(items)  # das Original bleibt unverändert

Drittens nutzt du `None` bewusst als Signal „nichts gefunden“: Die Funktion liefert den Treffer oder `None`. Der Aufrufer prüft mit `is None`:

In [ ]:
def find_customer(customers, name):
    for customer in customers:
        if customer.lower() == name.lower():
            return customer
    return None  # nichts gefunden


customers = ["Ada", "Linus", "Grace"]
for name in ["grace", "Tim"]:
    customer = find_customer(customers, name)
    if customer is None:
        print(f"{name}: kein Kunde gefunden")
    else:
        print(f"{name}: gefunden als {customer}")

**Kurzübung 2 (Zwischensumme ist None):** Der folgende Code soll die Summe eines Warenkorbs ausgeben, bricht aber mit einem `TypeError` ab. Welche Ausgabe erzeugt er vor dem Abbruch, und warum? Behebe den Fehler, sodass `line_total` ihr Ergebnis zurückgibt.

In [ ]:
def line_total(quantity, unit_price):
    print(quantity * unit_price)


cart = [(2, 4.50), (1, 3.80), (3, 2.20)]
total = 0
for quantity, unit_price in cart:
    total += line_total(quantity, unit_price)
print(f"Summe: {total:.2f} €")

In [ ]:
# Deine Lösung


## Standardwerte und Schlüsselwort-Argumente

Parameter können einen **Standardwert** haben. Beim Aufruf darfst du Argumente auch **mit Namen** übergeben; dann ist die Reihenfolge egal.

In [ ]:
def gross_price(net_price, vat_rate=0.19):
    return net_price * (1 + vat_rate)


print(gross_price(100))
print(gross_price(100, 0.07))
print(gross_price(vat_rate=0.07, net_price=100))

**Kurzübung 3 (Preise formatieren):** Schreibe `format_price(amount, currency="€", decimals=2)`, die einen Preis als Text liefert, etwa `"12.50 €"`. Rufe sie auf mit nur einem Betrag, mit `"$"` als Währung und mit `decimals=0` bei unveränderter Währung. Tipp: `f"{amount:.{decimals}f}"` setzt die Zahl der Nachkommastellen aus einer Variablen ein.

- **Untersuchen:** Welche Ausgabe erzeugt `format_price(12.5, 0)`, und warum?

In [ ]:
# Deine Lösung


> **Typischer Fehler:** Kein veränderbarer Standardwert wie `def f(items=[])`. Python legt die Liste nur einmal an, beim `def`; alle Aufrufe teilen sie sich. Nimm stattdessen `None` und lege die Liste in der Funktion an:

In [ ]:
def add_item(item, items=[]):
    items.append(item)
    return items


print(add_item("Milch"))
print(add_item("Brot"))  # ['Milch', 'Brot']: die Milch vom ersten Aufruf ist noch da


def add_item_fixed(item, items=None):
    if items is None:
        items = []  # bei jedem Aufruf eine neue Liste
    items.append(item)
    return items


print(add_item_fixed("Milch"))
print(add_item_fixed("Brot"))  # ['Brot']

## Mehrere Rückgabewerte

Eine Funktion kann ein Tupel zurückgeben, das du direkt entpackst:

In [ ]:
def min_max(numbers):
    return min(numbers), max(numbers)


lowest, highest = min_max([5, 2, 9, 1])
print(lowest, highest)

**Kurzübung 4 (Brutto aufteilen):** Schreibe `split_gross(gross_price, vat_rate=0.19)`, die aus einem Bruttopreis den Nettopreis und die enthaltene Mehrwertsteuer berechnet und beide zurückgibt. Entpacke das Ergebnis beim Aufruf in zwei Variablen und gib sie aus, für 119 € etwa `Netto 100.00 €, MwSt 19.00 €`. Prüfe auch 10,70 € mit 7 %. Tipp: Netto = Brutto / (1 + Steuersatz).

In [ ]:
# Deine Lösung


## Docstrings und Type Hints

Ein **Docstring** beschreibt, was die Funktion tut. **Type Hints** zeigen, welche Typen erwartet werden. Python prüft sie nicht, aber Editoren und Kollegen profitieren davon. Hinter dem Parameter steht sein Typ, hinter `->` der Typ des Ergebnisses. Für Collections steht der Typ der Elemente in eckigen Klammern:

- `list[str]` heißt „Liste von Strings“,
- `tuple[str, float]` heißt „Tupel aus Text und Zahl“,
- `str | None` heißt „ein String oder `None`“.

In [ ]:
def bmi(weight_kg: float, height_cm: float) -> float:
    """Berechnet den Body-Mass-Index aus Gewicht und Größe."""
    return weight_kg / (height_cm / 100) ** 2


help(bmi)

In [ ]:
# find_customer von oben, jetzt mit Type Hints und Docstring
def find_customer(customers: list[str], name: str) -> str | None:
    """Sucht einen Kunden ohne Rücksicht auf Groß- und Kleinschreibung; None, wenn er fehlt."""
    for customer in customers:
        if customer.lower() == name.lower():
            return customer
    return None


print(find_customer(["Ada", "Linus", "Grace"], "grace"))

**Kurzübung 5 (Versandkosten dokumentieren):** Ergänze in `shipping_cost` Type Hints und einen Docstring, der in einem Satz sagt, was die Funktion liefert. Zeige das Ergebnis mit `help(shipping_cost)`.

- **Untersuchen:** Python prüft Type Hints nicht. Welche Ausgabe erzeugt `shipping_cost(60, "ja")`, und warum?

In [ ]:
def shipping_cost(order_total, express=False):
    if order_total >= 50 and not express:
        return 0.0
    cost = 4.95
    if express:
        cost += 5.00
    return cost

In [ ]:
# Deine Lösung


## Platzhalter mit `pass`

Ein Block darf nicht leer sein. `pass` ist eine Anweisung, die nichts tut, praktisch für Funktionen, die du später schreibst.

In [ ]:
def send_report():
    pass  # TODO: später umsetzen

## Gültigkeitsbereich und Namensräume

Ein **Namensraum** (*namespace*) ordnet Namen ihren Objekten zu, ähnlich wie ein Dictionary. Python kennt drei Ebenen:

- **builtins:** die eingebauten Namen wie `print`, `len` und `round` (Notebook 02).
- **global:** alle Namen, die du im Notebook oder in einer Datei auf oberster Ebene anlegst.
- **lokal:** die Namen einer Funktion, also ihre Parameter und alles, was sie zuweist. Jeder Aufruf bekommt einen neuen lokalen Namensraum; nach `return` verschwindet er wieder.

Der **Gültigkeitsbereich** (*scope*) eines Namens ist der Teil des Programms, in dem der Name sichtbar ist. `locals()` zeigt den lokalen Namensraum eines Aufrufs:

In [ ]:
VAT_RATE = 0.19


def gross_price(net_price):
    tax = net_price * VAT_RATE
    print(locals())  # nur die lokalen Namen dieses Aufrufs
    return round(net_price + tax, 2)


print(gross_price(100))

Die Namensräume liegen ineinander wie Mengen. Trifft Python in der Funktion auf einen Namen, sucht es von innen nach außen: erst lokal, dann global, zuletzt in den builtins. Der erste Treffer gewinnt; findet Python den Namen nirgends, meldet es einen `NameError`.

![Drei ineinanderliegende Mengen: builtins mit print, len und round enthält global mit VAT_RATE und gross_price, das enthält lokal für den Aufruf gross_price(100) mit net_price = 100 und tax = 19.0. Darunter die Suche von innen nach außen: VAT_RATE findet Python global, round in den builtins, tax draußen im Notebook nirgends, also NameError.](../assets/07_scopes.png)

> Liegt eine Funktion in einer anderen, sucht Python zwischen lokal und global noch in der äußeren Funktion. Die ganze Regel heißt kurz **LEGB**: Local, Enclosing, Global, Builtins.

Draußen im Notebook beginnt die Suche erst bei global. Den lokalen Namensraum des Aufrufs gibt es dann schon nicht mehr:

In [ ]:
print(tax)  # tax gab es nur während des Aufrufs

**Lesen geht, zuweisen nicht:** Eine Funktion darf globale Namen lesen, wie `VAT_RATE` oben. Weist sie einem Namen aber irgendwo etwas zu, ist dieser Name für die **ganze** Funktion lokal. Das überrascht bei `+=`:

In [ ]:
visitors = 0


def count_visit():
    visitors += 1  # Zuweisung: visitors ist jetzt lokal und hat noch keinen Wert


count_visit()

`visitors += 1` bedeutet `visitors = visitors + 1`. Wegen der Zuweisung ist `visitors` in der Funktion lokal, und rechts soll Python den lokalen Namen lesen, bevor er einen Wert hat. Besser: Gib den Wert als Parameter herein und das Ergebnis mit `return` zurück:

In [ ]:
def count_visit(visitors):
    return visitors + 1


visitors = count_visit(visitors)
visitors = count_visit(visitors)
print(visitors)  # 2

> Mit `global visitors` in der Funktion könntest du den globalen Namen trotzdem ändern. Lass das: Dann hängt die Funktion von einem Zustand außerhalb ab, und beim Aufruf sieht niemand, was sie verändert (siehe „Gute Funktionen“ unten).

**Kurzübung 6 (Versand aufschlagen):** Sage die Ausgabe des folgenden Codes vorher, bevor du ihn ausführst: Welchen Wert hat `total` am Ende, und warum? Ändere dann den Aufruf so, dass `total` danach den Betrag mit Versand enthält.

In [ ]:
total = 100


def add_shipping(total):
    total = total + 4.95
    return total


add_shipping(total)
print(total)

In [ ]:
# Deine Lösung


Anders bei veränderbaren Objekten: Eine übergebene Liste ist **dieselbe** Liste wie draußen (Notebook 06, „Zuweisung kopiert nicht“). Ändert die Funktion sie mit einer Methode wie `append`, ändert sie das Original:

In [ ]:
def add_bonus(scores):
    scores.append(100)


my_scores = [80, 90]
add_bonus(my_scores)
my_scores

![Der globale Name my_scores und der lokale Parameter scores in add_bonus verweisen auf dieselbe Liste; scores.append(100) hängt 100 an diese eine Liste an.](../assets/07_namensraeume.png)

## Beliebig viele Argumente

Mit `*` vor einem Parameter nimmt eine Funktion beliebig viele Positionsargumente an. Sie landen in einem **Tupel**. Du kennst das von `print()`, das beliebig viele Werte ausgibt:

In [ ]:
def average(*numbers: float) -> float:
    return sum(numbers) / len(numbers)


print(average(1, 2, 3))
print(average(10, 20))

Mit `**` sammelst du beliebig viele **Schlüsselwort-Argumente** in einem Dictionary. Oft heißen die Parameter `*args` und `**kwargs`; die Namen sind nur Konvention, entscheidend sind `*` und `**`:

In [ ]:
def describe(name, **details):
    print(details)  # ein Dictionary aus Namen und Werten
    text = name
    for key, value in details.items():
        text += f", {key}: {value}"
    return text


describe("Ada", born=1815, city="London")

Umgekehrt packt `*` beim **Aufruf** eine Liste in einzelne Argumente aus:

In [ ]:
values = [4, 8, 15]
print(average(*values))  # wie average(4, 8, 15)

**Kurzübung 7 (Artikel zählen):** Sage für jede `print`-Zeile die Ausgabe vorher und begründe sie, bevor du den Code ausführst. Warum liefert `count_articles(cart)` nicht 3?

In [ ]:
def count_articles(*articles):
    return len(articles)


cart = ["Milch", "Brot", "Eier"]
print(count_articles("Milch", "Brot"))
print(count_articles())
print(count_articles(cart))
print(count_articles(*cart))

In [ ]:
# Deine Lösung


## Funktionen sind Werte: Lambdas

Eine Funktion ist ein Objekt wie jedes andere (Notebook 02). Du kannst sie einer Variablen zuweisen oder einer anderen Funktion übergeben, **ohne** sie aufzurufen, also ohne Klammern:

In [ ]:
def shout(text: str) -> str:
    return text.upper() + "!"


action = shout  # ohne Klammern: die Funktion selbst, kein Aufruf
print(action("hallo"))
print(list(map(shout, ["ja", "nein"])))  # map ruft shout für jedes Element auf

Genau das nutzt `sorted(key=…)` aus Notebook 06: Du übergibst eine Funktion, und `sorted` ruft sie für jedes Element auf. Für solche kleinen Einmal-Funktionen gibt es die Kurzform **`lambda`**:

```
lambda parameter: ausdruck
```

Ein Lambda hat keinen Namen und besteht aus genau **einem Ausdruck**. Dessen Wert gibt es zurück, ganz ohne `return`. `max()` und `min()` verstehen `key=` genauso wie `sorted()`:

In [ ]:
people = [("Ada", 36), ("Linus", 55), ("Grace", 85)]

by_age = sorted(people, key=lambda person: person[1])
oldest = max(people, key=lambda person: person[1])
print(by_age)
print(oldest)

**Kurzübung 8 (Sortierung vorhersagen):** Welche Ausgabe erzeugen die beiden Zeilen, und warum? Schreibe deine Vorhersage auf und führe den Code dann aus. Was ändert sich, wenn du `key` weglässt?

In [ ]:
words = ["kiwi", "Banane", "apfel"]
print(sorted(words, key=lambda word: word.lower()))
print(sorted(words, key=lambda word: word[-1]))

In [ ]:
# Deine Lösung


> Weise ein Lambda nicht einer Variablen zu (`square = lambda x: x * x`). Dafür ist `def` lesbarer, und die Funktion hat einen Namen, der im Traceback erscheint. Lambdas gehören direkt dorthin, wo eine kleine Funktion übergeben wird.

## Rekursion

Eine Funktion darf sich **selbst aufrufen**. Das heißt Rekursion. Sie passt zu Problemen, die sich in eine kleinere Version desselben Problems zerlegen lassen.

Beispiel Fakultät: 5! = 5 · 4 · 3 · 2 · 1, also 5! = 5 · 4!. Jede rekursive Funktion braucht zwei Teile:

1. **Abbruchbedingung** (Basisfall): Hier ist die Antwort direkt klar, 1! = 1.
2. **Rekursiver Aufruf** mit einem **kleineren** Problem: n! = n · (n − 1)!

In [ ]:
def factorial(n: int) -> int:
    if n <= 1:  # Abbruchbedingung
        return 1
    return n * factorial(n - 1)  # kleineres Problem


factorial(5)

So läuft `factorial(3)` ab. Jeder Aufruf wartet, bis der innere fertig ist:

![factorial(3) ruft factorial(2) auf, das ruft factorial(1) auf; die Abbruchbedingung liefert 1, damit rechnet factorial(2) 2 * 1 = 2 und factorial(3) 3 * 2 = 6.](../assets/07_rekursion.png)

Fehlt die Abbruchbedingung, ruft sich die Funktion endlos auf. Python bricht nach etwa 1000 Ebenen mit einem `RecursionError` ab:

In [ ]:
def broken_factorial(n: int) -> int:
    return n * broken_factorial(n - 1)  # Abbruchbedingung vergessen


broken_factorial(5)

Richtig stark ist Rekursion bei **verschachtelten** Strukturen, deren Tiefe du vorher nicht kennst: Ordner mit Unterordnern, Menüs mit Untermenüs, Kommentare mit Antworten. Hier ein Menü als verschachtelte Liste. `isinstance(item, list)` prüft, ob ein Element selbst eine Liste ist:

In [ ]:
def count_items(nested: list) -> int:
    total = 0
    for item in nested:
        if isinstance(item, list):
            total += count_items(item)  # Unterliste: rekursiv zählen
        else:
            total += 1
    return total


menu = ["Start", ["Datei", ["Öffnen", "Speichern"]], ["Hilfe", ["Über"]]]
count_items(menu)

**Kurzübung 9 (Menü zählen):** Sage vorher, welchen Wert `count_items(["Start", ["Datei"], []])` liefert und wie oft `count_items` dabei insgesamt aufgerufen wird. Begründe beides und prüfe danach mit dem Aufruf.

In [ ]:
# Deine Lösung


> Für einfache Wiederholungen wie die Fakultät ist eine Schleife in Python meist die bessere Wahl. Rekursion lohnt sich, wenn die Daten selbst verschachtelt sind.

## Generatoren mit `yield`

### Was `for` unter der Haube macht

Listen, Tupel, Strings, Dictionaries, Sets, `range()`, Dateien: Über all das kann `for` laufen. Solche Objekte heißen **iterierbar** (*iterable*). Wie funktioniert das?

`for` holt sich mit `iter()` einen **Iterator**. Das ist ein Objekt, das sich merkt, wo es gerade steht, und mit `next()` das nächste Element liefert:

In [ ]:
letters = ["a", "b", "c"]
iterator = iter(letters)
print(next(iterator))
print(next(iterator))
print(next(iterator))

Sind alle Elemente geliefert, löst `next()` eine `StopIteration` aus:

In [ ]:
next(iterator)

`for` ruft `next()` auf, bis `StopIteration` kommt, und hört dann still auf. In einer `for`-Schleife siehst du diesen Fehler deshalb nie.

**Wichtig:** Ein Iterator ist nach einem Durchlauf **verbraucht**. Viele eingebaute Funktionen liefern Iteratoren statt Listen, zum Beispiel `zip()` (verbindet Folgen paarweise), `map()` (siehe oben), `filter()` und `reversed()`. Das spart Speicher, kann aber überraschen:

In [ ]:
pairs = zip(["Ada", "Grace"], [1815, 1906])
print(list(pairs))
print(list(pairs))  # leer: der Iterator ist schon verbraucht

Brauchst du die Werte mehrmals, wandle sie einmal mit `list()` um.

**Kurzübung 10 (Ergebnisliste):** Der folgende Code soll erst die Zahl der Ergebnisse und dann jedes Ergebnis einzeln ausgeben. Welche Ausgabe erzeugt er, und warum fehlen die einzelnen Zeilen? Behebe den Fehler.

In [ ]:
names = ["Ada", "Tim", "Lili"]
points = [87, 92, 78]

results = zip(names, points)
print(len(list(results)), "Ergebnisse")
for name, score in results:
    print(f"{name}: {score} Punkte")

In [ ]:
# Deine Lösung


### Ein Generator ist ein selbst geschriebener Iterator

Eine Funktion mit `return` liefert **einen** Wert und ist fertig. Eine Funktion mit **`yield`** liefert Werte **nacheinander**: Sie gibt einen Wert ab, pausiert und macht beim nächsten Mal genau dort weiter. So eine Funktion heißt **Generator**.

In [ ]:
def countdown(start):
    while start > 0:
        yield start
        start -= 1
    print("Start!")


for number in countdown(3):
    print(number)

`for` holt sich vom Generator mit `next()` Wert für Wert. Der Code im Generator läuft dabei immer nur bis zum nächsten `yield` und erst, wenn ein Wert angefordert wird. Deshalb erscheint `Start!` erst, nachdem die Schleife die letzte Zahl geholt hat.

![Die for-Schleife ruft viermal next() beim Generator countdown(3) auf: Er liefert per yield 3, 2 und 1 und pausiert jeweils; beim vierten Aufruf endet die while-Schleife, er gibt Start! aus und meldet StopIteration, worauf die for-Schleife still endet.](../assets/07_generator.png)

Der große Vorteil: Ein Generator berechnet jeden Wert erst bei Bedarf und hält nicht alle gleichzeitig im Speicher. Er kann sogar **unendlich** viele Werte liefern; der Aufrufer entscheidet, wann Schluss ist:

In [ ]:
def even_numbers():
    number = 0
    while True:
        yield number
        number += 2


for number in even_numbers():
    if number > 10:
        break
    print(number, end=" ")

Für einfache Fälle gibt es den **Generator-Ausdruck**: Er sieht aus wie eine List Comprehension, aber mit runden Klammern. Er baut keine Liste, sondern liefert die Werte nacheinander. Ideal als Argument für `sum()`, `max()` oder `"".join()`; die Klammern des Funktionsaufrufs genügen dann:

In [ ]:
total = sum(n ** 2 for n in range(1_000_000))
total

**Kurzübung 11 (Rechnungsnummern):** Schreibe einen Generator `invoice_numbers(start=1)`, der endlos Rechnungsnummern im Format `R-0001`, `R-0002`, … liefert. Hole dir mit `next()` die ersten drei Nummern. Berechne außerdem mit einem Generator-Ausdruck in `sum()` die Summe aller Beträge in `amounts` über 100 € (hier 360 €).

In [ ]:
amounts = [89.90, 240.00, 15.50, 120.00]

In [ ]:
# Deine Lösung


## Gute Funktionen

- **Ein** Zweck pro Funktion, Name als Verb: `calculate_total`, `read_file`
- Lieber Werte zurückgeben als drucken
- Kurz halten: passt sie nicht auf einen Bildschirm, aufteilen
- Keine globalen Variablen verändern, alles über Parameter hereingeben

> Wie du Funktionen in eigene Dateien auslagerst und importierst, zeigt Notebook 08.

## Übungen

**Übung 1 (Eigene Listenfunktionen, leicht):** Schreibe nach dem Muster von `my_sum` aus Kurzübung 1 zwei weitere Funktionen, ohne die eingebauten `max` und `reversed` und ohne Slicing mit `[::-1]`:

- `my_max(numbers)` liefert die größte Zahl.
- `reverse_list(items)` liefert eine neue Liste in umgekehrter Reihenfolge; die übergebene Liste bleibt unverändert.

Teste sie mit `prices` und vergleiche die Ergebnisse mit `max(prices)` und `prices[::-1]`. Für `reverse_list` gibt es mehrere Wege: Probiere zwei aus und entscheide, welchen du nimmst, und warum. Was liefert `my_max([])`, und was `max([])`?

In [ ]:
prices = [4.5, 12.0, 3.2, 8.9]

In [ ]:
# Deine Lösung


**Übung 2 (Rechnungszeilen mit Kopierfehler, leicht):** Der folgende Code gibt drei fast gleiche Rechnungszeilen aus; beim Kopieren ist ein Fehler hineingeraten. Finde den Fehler, lagere die Logik in `format_invoice_line(product, quantity, unit_price)` aus und gib zusätzlich eine Summenzeile aus.

In [ ]:
print(f"{'Kaffee':<10} {2:>3} x {4.50:>6.2f} € = {2 * 4.50:>7.2f} €")
print(f"{'Kuchen':<10} {1:>3} x {3.80:>6.2f} € = {2 * 3.80:>7.2f} €")
print(f"{'Wasser':<10} {3:>3} x {2.20:>6.2f} € = {3 * 2.20:>7.2f} €")

In [ ]:
# Deine Lösung


**Übung 3 (Preis nachschlagen, leicht):** Schreibe nach dem Muster von `find_customer` die Funktion `find_price(products, name)`, die in `products` den Preis eines Produkts sucht, ohne auf Groß- und Kleinschreibung zu achten, und `None` liefert, wenn es das Produkt nicht gibt. Ergänze Type Hints (Tipp: `products` ist eine Liste von Tupeln aus Text und Zahl, siehe Abschnitt „Docstrings und Type Hints“). Gib dann für jedes Produkt in `order` den Preis oder „nicht im Sortiment“ aus. Erwartet: `kaffee: 2.90 €`, `Leitungswasser: 0.00 €`, `Kuchen: nicht im Sortiment`.

- **Untersuchen:** Was geht schief, wenn du statt `if price is None:` nur `if not price:` schreibst? Probiere es mit dem Leitungswasser aus.

In [ ]:
products = [("Kaffee", 2.90), ("Tee", 2.50), ("Leitungswasser", 0.0)]
order = ["kaffee", "Leitungswasser", "Kuchen"]

In [ ]:
# Deine Lösung


**Übung 4 (Arbeitszeit erfassen, mittel):** Für die Zeiterfassung sollen Arbeitszeiten aus Beginn und Ende berechnet werden, beide als Text wie `"08:15"`.

- Schreibe `to_minutes(time)`, die aus `"08:15"` die Minuten seit Mitternacht macht, hier 495.
- Schreibe `worked_hours(start, end, break_minutes=30)`, die mit `to_minutes` die Arbeitszeit in Stunden abzüglich der Pause liefert. Kontrolle: `worked_hours("08:00", "16:30")` liefert 8.0, `worked_hours("09:15", "13:45", break_minutes=0)` liefert 4.5.
- Gib für jeden Tag in `week` die Stunden aus, etwa `Mo: 8.00 h`, und am Ende die Wochensumme (hier 20.25 h).

Für `to_minutes` gibt es zwei Wege: `split(":")` oder Slicing mit `time[:2]` und `time[3:]`. Welchen nimmst du, und warum? Probiere beide auch mit `"8:15"` aus.

In [ ]:
week = [("Mo", "08:00", "16:30"), ("Di", "08:15", "17:00"), ("Mi", "09:15", "13:45")]

In [ ]:
# Deine Lösung


**Übung 5 (Passwortregeln als Funktion, mittel):** Der Startcode ist die Passwortprüfung aus 05, Übung 4: Er prüft ein einziges Passwort auf Länge, Ziffer, Groß- und Kleinbuchstaben und gibt die Mängel direkt aus. Mach daraus eine wiederverwendbare Funktion und ergänze eine neue Regel.

- Schreibe `password_problems(password, min_length=12)`, die die Mängel als Liste von Texten zurückgibt, statt sie auszugeben; eine leere Liste heißt „sicher“. Ergänze Type Hints und einen Docstring. Kontrolle: `password_problems("Sommer2026")` liefert `['kürzer als 12 Zeichen']`, `password_problems("sommer", min_length=8)` liefert `['kürzer als 8 Zeichen', 'Ziffer fehlt', 'Großbuchstabe fehlt']`.
- Neue Regel: Das Passwort braucht ein Sonderzeichen, also ein Zeichen, das weder Buchstabe noch Ziffer ist (`not char.isalnum()`). Prüfe das mit `any()`: `any(...)` liefert `True`, sobald mindestens ein Wert wahr ist, etwa `any(char.isdigit() for char in password)`. Danach liefert `password_problems("Sommer2026")` die Liste `['kürzer als 12 Zeichen', 'Sonderzeichen fehlt']` und `password_problems("Sommer-Abend-2026")` eine leere Liste.
- **Untersuchen:** Schreibe auch die anderen Prüfungen mit `any()` statt mit Merkern. Warum wäre diese Schleife in der Funktion falsch?

  ```python
  for char in password:
      if char.isdigit():
          return True
      else:
          return False
  ```

In [ ]:
password = "Sommer2026"
has_digit = False
has_upper = False
has_lower = False
for char in password:
    if char.isdigit():
        has_digit = True
    elif char.isupper():
        has_upper = True
    elif char.islower():
        has_lower = True
if len(password) < 12:
    print("kürzer als 12 Zeichen")
if not has_digit:
    print("Ziffer fehlt")
if not has_upper:
    print("Großbuchstabe fehlt")
if not has_lower:
    print("Kleinbuchstabe fehlt")

In [ ]:
# Deine Lösung


**Übung 6 (Kontakte sortieren, mittel):** `contacts` enthält Tupel aus Vorname, Nachname und Stadt.

- (a) Sortiere die Kontakte nach Nachnamen. Schreibe den Schlüssel zuerst als eigene Funktion mit `def`, dann als `lambda`. Welche Fassung liest sich besser, und wann nimmst du welche?
- (b) Sortiere die Kontakte nach Stadt und bei gleicher Stadt nach Nachnamen. Tipp: Ein Tupel als Schlüssel, `key=lambda c: (c[2], c[1])`, sortiert erst nach dem ersten Wert und bei Gleichstand nach dem zweiten. Erwartet: Liskov (Boston), Lovelace und Turing (London), Hopper (New York).
- (c) Finde mit `max` und `key=` den Kontakt mit dem längsten Nachnamen. Erwartet: `('Ada', 'Lovelace', 'London')`.

In [ ]:
contacts = [
    ("Grace", "Hopper", "New York"),
    ("Ada", "Lovelace", "London"),
    ("Alan", "Turing", "London"),
    ("Barbara", "Liskov", "Boston"),
]

In [ ]:
# Deine Lösung


**Übung 7 (Ordnergröße, schwer):** Ein Ordner ist ein Dictionary aus Namen und entweder Dateigröße in KB (`int`) oder Unterordner (`dict`). Der Startcode berechnet eine Ordnergröße. Sage vorher, was er für `home` liefert, führe den Code dann aus und finde heraus, was fehlt. Formuliere ihn rekursiv, sodass er alle Unterordner mitzählt (erwartet: 6324 KB).

- **Untersuchen:** Welche Dateien zählt der Startcode nicht? Was müsste für einen Unterordner passieren?
- **Ändern:** Rufe `folder_size` für Unterordner rekursiv auf.

Schreibe danach `print_tree(folder, indent=0)`, die den Ordner als eingerückten Baum ausgibt: Dateien mit Größe, Ordner mit `/` am Ende, und für jeden Unterordner ruft sich `print_tree` mit `indent + 2` selbst auf. Tipp: `" " * indent` erzeugt die Einrückung. Erwartet:

```
cv.pdf (120 KB)
Fotos/
  urlaub.jpg (3400 KB)
  2026/
    a.jpg (2800 KB)
notes.txt (4 KB)
```

In [ ]:
def folder_size(folder):
    total = 0
    for content in folder.values():
        if isinstance(content, int):
            total += content
    return total


home = {
    "cv.pdf": 120,
    "Fotos": {"urlaub.jpg": 3400, "2026": {"a.jpg": 2800}},
    "notes.txt": 4,
}
print(folder_size(home), "KB")

In [ ]:
# Deine Lösung


**Übung 8 (Rabattcode: in einem Satz erklären, mittel, für Schnelle):** Lies `f_original` und beschreibe in **einem Satz**, was sie tut. Öffne einen neuen Chat ohne den Tutor-Prompt aus Notebook 00 und gib nur den Satz ein, mit „Schreib eine Python-Funktion `f(total, code)`, die …“. Kopiere die Antwort in die Zelle darunter und führe danach die Tests in der Zelle darauf aus: Sie prüfen das `f` der KI. Schlagen sie fehl, fehlte im Satz ein Randfall. Vergleiche danach: Wie geht die KI vor, wie das Original? Und was liefert das Original, wenn du `case _` weglässt? Frag die KI nach jeder Zeile, die du nicht kennst.

Ohne KI-Zugang: Prüfe die typische KI-Antwort aus der Lösung.

In [ ]:
def f_original(total, code):
    match code:
        case "SPAR10":
            return round(total * 0.9, 2)
        case "MINUS5" if total >= 20:
            return total - 5
        case _:
            return total

In [ ]:
# Deine Lösung


In [ ]:
# Tests: Sie prüfen f aus der Antwort der KI, nicht f_original.
# Ein NameError heißt: Die Antwort der KI fehlt noch oder heißt nicht f.
assert f(42.0, "SPAR10") == f_original(42.0, "SPAR10") == 37.8
assert f(25.0, "MINUS5") == 20.0
assert f(15.0, "MINUS5") == 15.0  # Randfall: unter 20 € kein Abzug
assert f(30.0, "GRATIS") == 30.0  # Randfall: unbekannter Code ändert nichts

## Zusammenfassung

- `def name(parameter):` definiert, `return` liefert das Ergebnis.
- Ohne `return` liefert eine Funktion `None`, ebenso Methoden wie `sort()`; `-> str | None` und `if x is None:` signalisieren „nichts gefunden“.
- Standardwerte `def f(x, rate=0.19)` und Aufrufe mit Namen `f(rate=0.07, x=1)`; kein veränderbarer Standardwert wie `items=[]`, sondern `items=None`.
- Namensräume: builtins, global und lokal liegen ineinander; Python sucht von innen nach außen (LEGB), der erste Treffer gewinnt. Namen, die in einer Funktion entstehen, auch Parameter, gibt es nur während des Aufrufs. Eine Zuweisung macht einen Namen für die ganze Funktion lokal; Ergebnisse holst du mit `return` heraus statt mit `global`. Eine übergebene Liste ist dagegen dieselbe Liste: Ändert die Funktion sie, ändert sie das Original.
- `*args` sammelt beliebig viele Argumente in einem Tupel, `**kwargs` in einem Dictionary; `f(*liste)` packt beim Aufruf aus.
- Docstring und Type Hints machen Funktionen selbsterklärend.
- `pass` als Platzhalter für leere Blöcke.
- Funktionen sind Werte: übergeben ohne Klammern. `lambda x: ausdruck` für kleine Einmal-Funktionen, etwa als `key`.
- Rekursion: Abbruchbedingung plus Aufruf mit kleinerem Problem. Stark bei verschachtelten Daten.
- `for` holt sich mit `iter()` einen Iterator und ruft `next()` auf, bis `StopIteration` kommt. Iteratoren wie `zip()` sind nach einem Durchlauf verbraucht.
- Generatoren liefern mit `yield` Werte nacheinander und erst bei Bedarf; `(x for x in …)` ist die Kurzform.
- Aus den Übungen: `any(...)` prüft, ob mindestens ein Wert wahr ist.